# 03 — Perfis de pedidos com K-Means de raiz
**Objetivo proposto:** agrupar pedidos com características financeiras e do agregado semelhantes.
K-Means e inicialização k-means++ estão em libs/kmeans.py; silhouette está em libs/metrics.py.

TARGET e SK_ID_CURR não entram nas distâncias.
Não assumimos antecipadamente que os grupos são níveis de risco.

**Estado:** implementação inicial, sem resultados reais publicados.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from libs.data import FEATURES, load_applications

SEED = 42
# Mantém downloads explícitos: a autenticação e as regras são geridas no Kaggle.
DOWNLOAD = False
applications, dictionary = load_applications(download=DOWNLOAD)
print(f"Pedidos: {len(applications):,}; colunas: {len(applications.columns)}")

# Guardar estas informações com os resultados permite repetir a experiência.
display(pd.DataFrame(applications.attrs["source_files"]))
print({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__})

## Preparação
Selecionamos até 20 000 pedidos aleatórios. Aplicamos log1p aos montantes e depois
normalização por média/desvio padrão, para reduzir domínio por escala e assimetria.
As medianas tratam valores em falta. Não há remoção automática de extremos.

Esta análise é descritiva: a preparação é ajustada na amostra de clustering.
Caso se pretenda avaliar generalização, será necessária uma separação própria.

In [ ]:
from libs.data import NumericPreparation
from libs.kmeans import KMeans
from libs.metrics import silhouette_score
from time import perf_counter

MAX_ROWS = 20_000
sample = applications.sample(n=min(MAX_ROWS, len(applications)), random_state=SEED).copy()
preparation = NumericPreparation(scale=True, log_amounts=True).fit(sample)
X = preparation.transform(sample)
if len(X) < 10:
    raise ValueError("A experiência de K=2 a 8 requer pelo menos dez pedidos.")
# A mesma subamostra em todos os K torna a comparação mais justa e limita o custo quadrático.
evaluation_indices = np.random.default_rng(SEED).choice(len(X), size=min(500, len(X)), replace=False)

## Comparar K e inicializações
Testamos K de 2 a 8 e cinco inicializações por K. O algoritmo mantém a menor inércia.
A silhouette é calculada numa subamostra fixa de até 500 pontos, não na base completa.
A sua melhor pontuação é uma proposta de K, a confrontar com inércia, tamanhos e interpretação.

In [ ]:
models, results = {}, []
started = perf_counter()
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=5, max_iter=100, seed=SEED).fit(X)
    models[k] = model
    try:
        score = silhouette_score(X[evaluation_indices], model.labels_[evaluation_indices])
        note = ""
    except ValueError as error:
        score, note = np.nan, str(error)
    results.append({
        "K": k, "inertia": model.inertia_, "silhouette_amostra": score,
        "iterações": model.n_iter_, "convergiu": model.converged_, "nota": note,
    })
summary = pd.DataFrame(results).set_index("K")
display(summary)
print(f"Tempo: {perf_counter() - started:.1f} s")
summary[["inertia", "silhouette_amostra"]].plot(subplots=True, marker="o", figsize=(8, 6))
plt.tight_layout()
plt.show()
eligible = summary.loc[summary["convergiu"], "silhouette_amostra"].dropna()
if eligible.empty:
    raise RuntimeError("Nenhuma solução convergente com silhouette válida; rever configuração.")
candidate_k = int(eligible.idxmax())
print("K candidato pela silhouette da amostra:", candidate_k)

## Estabilidade e perfis
Comparamos inércia, silhouette e tamanhos em sementes adicionais.
Isto é um diagnóstico inicial; não prova estabilidade das atribuições individuais.

Descrevemos perfis através das medianas nas unidades originais.
Só depois de formar os grupos calculamos a proporção observada de TARGET=1.

In [ ]:
stability = []
for seed in [SEED, SEED + 1, SEED + 2]:
    model = KMeans(n_clusters=candidate_k, n_init=5, seed=seed).fit(X)
    try:
        score = silhouette_score(X[evaluation_indices], model.labels_[evaluation_indices])
    except ValueError:
        score = np.nan
    stability.append({
        "semente": seed, "inertia": model.inertia_, "silhouette_amostra": score,
        "convergiu": model.converged_,
        "tamanhos_ordenados": sorted(np.bincount(model.labels_).tolist()),
    })
display(pd.DataFrame(stability))
sample["cluster"] = models[candidate_k].labels_
display(sample.groupby("cluster")[FEATURES].median())
display(sample.groupby("cluster").agg(
    pedidos=("SK_ID_CURR", "size"),
    proporção_TARGET_1=("TARGET", "mean"),
))

## Interpretação — preencher após executar
- Como se comparam silhouette, inércia e interpretação dos vários K?
- Existem grupos muito pequenos ou dominados por valores extremos?
- Os resultados variam entre sementes?
- Que perfis são descritos pelas medianas? Que variáveis dominam?
- O que muda ao retirar variáveis redundantes ou rever transformações?

Não atribuir nomes como «baixo risco» ou «alto risco» sem fundamentação.
A proporção de TARGET=1 é uma comparação posterior, não uma variável de formação dos grupos.